# ArNet2 - Preprocessing and Model Training

This notebook demonstrates the workflow for:
1. Preprocessing R-peaks annotation data from the **SHDB-AF** dataset.
2. Training the **ArNet2** model for classification of **Atrial Fibrillation (AF)**.

---

### Disclaimer:

The **preprocessed data** in this notebook is **constructed from only 98 patients** from the **SHDB-AF** dataset. In real-world applications, **more diverse training data** (from multiple patients with varying AF conditions) is required to build a robust model. Using data from a very low numbers of patients can result in overfitting and limit the generalization ability of the model. Therefore, for a model to perform well on unseen data, it is essential to train on a **larger and more varied dataset**.

### Dataset Overview:

The **SHDB-AF** dataset consists of ECG signals, each annotated with:
- **R-peak annotations**: The location of R-peaks in the ECG signal.
- **AF labels per peak**: Whether the peak is associated with **Atrial Fibrillation (AF)**.
- **Overall patient-level label**: The AF status of the patient (e.g., **PAF**: Paroxysmal AF, **Persistent AF**, **Non-AF**).

This dataset will be used to demonstrate how to preprocess the data and train the **ArNet2** model.


### 1. Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import wfdb
import pickle
import subprocess
import nb_utils
from tqdm.notebook import tqdm  # Import tqdm for Jupyter Notebooks

### 2. Setting Up Paths and Directories

Before proceeding, we set up the paths where data will be downloaded and processed:

In [2]:
# Setting up the paths
data_path = '.././physionet_data'  # Where you'll download the PhysioNet dataset
output_path = '.././data'  # Where to save the processed data
os.makedirs(data_path, exist_ok=True)
os.makedirs(output_path, exist_ok=True)

### 3. Download Data

In this section, we define a function to download necessary files from PhysioNet. These files include the ECG signal and annotations:

In [3]:
# Download annotation files and additional data
nb_utils.download_file(f'https://physionet.org/files/shdb-af/1.0.1/AdditionalData.csv', f'{data_path}/AdditionalData.csv')

additionaldata = pd.read_csv(f'{data_path}/AdditionalData.csv')
record_names = additionaldata.Data_ID.astype(str).str.zfill(3)

In [4]:
for record_name in tqdm(record_names, desc="Downloading ECG records", leave=False):
    filepath = f'{data_path}/{record_name}'
    url = f'https://physionet.org/files/shdb-af/1.0.1/{record_name}'
    #  download records and annotations
    if nb_utils.url_exists(f"{url}.atr") and not os.path.exists(f"{filepath}.atr"):
        nb_utils.download_file(f'{url}.atr', f'{filepath}.atr')
        nb_utils.download_file(f'{url}.qrs', f'{filepath}.qrs')

### 4. Explanation of SHDB-AF Dataset

Key Components of SHDB-AF:

- ECG signals: These are the raw ECG waveform data recorded over time (.dat files).

- R-peak annotations: Each R-peak is labeled to indicate whether it corresponds to an AF event (1) or non-AF (0) (.atr files).

- Overall patient label: Each patient has a label indicating their overall AF status: PAF (Paroxysmal AF), Persistent AF, or Non-AF.

### 5. Preprocess ECG Data and Create Windows

Now, we preprocess the ECG data by extracting R-peaks and calculating RR intervals. We will then create fixed-length windows (e.g., 60-beat windows) and associate labels for training.

Preprocessing Steps:

- Extract RR intervals: Calculate the time difference between consecutive R-peaks.

- Create windows: Split the R-peaks data into 60-beat windows.

- Process Preceding Windows: Calculate preceding windows based on the RR intervals. feature is used during training to capture temporal information in the data.

- Assign labels: Each window will be labeled based on the presence of AF in the peaks. The AF label is 1 if the more than half of the R-peaks in the window are labeled as AF, and 0 otherwise.

In [5]:
# Load patient data
additionaldata = pd.read_csv(f'{data_path}/AdditionalData.csv')
record_names = additionaldata.Data_ID.astype(str).str.zfill(3)

In [6]:
global_label_dict = {'non-AF': 0, 'PAF': 1, 'PerAF': 3}

# Extract feature dicts and lists
windows_rr = {}  # dict to collect per-record features windows rr
windows_ts = {}  # dict to collect per-record features windows ts
prec_windows = {}  # dict to collect per-record preceding windows
global_labels = {}  # dict to collect per-record AF global labels
all_y_df = []  # list to collect per-record true labels per window
for record_name in tqdm(record_names, desc="Processing ECG records", leave=False):
    filepath = f"{data_path}/{record_name}"

    try:
        if not os.path.exists(f"{filepath}.atr"):
            continue

        else:
            annotation = wfdb.rdann(filepath, 'atr')

            # Load annotations (e.g., R-peaks)
            r_peaks = annotation.sample

            # Skip files with too few beats
            if len(r_peaks) < 2:
                print(f"Skipping {record_name}: not enough peaks ({len(r_peaks)})")
                continue

            # Calculate RR intervals and convert to seconds
            rr_intervals = np.diff(r_peaks).astype(np.float32)  # Calculate RR intervals in samples
            rr_time = r_peaks[1:] / annotation.fs  # Convert sample indices to time (seconds)
            rr_data = rr_intervals / annotation.fs  # RR intervals in seconds

            # Define window size: 60 beats per window (you can adjust this)
            window_size = 60  # 60 beats per window
            num_windows = len(rr_data) // window_size  # Number of 60-beat windows

            # Split the RR data into windows
            windows_rr[record_name] = rr_data[:num_windows * window_size].reshape(num_windows, window_size)
            windows_ts[record_name] = rr_time[:num_windows * window_size].reshape(num_windows, window_size)

            #  compute preceding windows per recording
            prec_windows[record_name] = nb_utils.calc_preceding_windows(r_peaks, annotation.fs, window_size)

            #  derive y labels
            y = nb_utils.calc_y(annotation.aux_note, window_size)[:len(windows_rr[record_name])]
            y_df = pd.DataFrame({
                'true_label': y,
                'prec_window': np.arange(y.size, dtype=int),
                'patient_id': record_name,
            })
            # Sanity check
            if len(windows_rr[record_name]) != len(y) or len(windows_ts[record_name]) != len(y):
                print(f"Length mismatch for patient_id={record_name}, skipping.")
                continue
                
            # Append to the lists
            all_y_df.append(y_df)

            # Generate global labels based on the patient diagnosis
            matching_row = additionaldata[additionaldata['Data_ID'].astype(str).str.zfill(3).eq(record_name)]
            global_labels[record_name] = np.repeat(global_label_dict[matching_row.AF_Type.iloc[0]], num_windows)
    except Exception as e:
        print(f"Error processing {record_name}: {e}")
        continue

# Concatenate all DataFrames into one
combined_y_df = pd.concat(all_y_df, ignore_index=True)

print(f"Processed overall {combined_y_df.patient_id.nunique()} patients ..")

Processing ECG records:   0%|          | 0/128 [00:00<?, ?it/s]

Processed overall 98 patients ..


### 8. Create Dataset for Model Training

Now we prepare the dataset by concatenating all features into a single matrix X and the labels into the y vector. We'll also include additional patient-level features such as the patient ID and global label.

In [7]:
X_all = []
y_all = []
timestamp_all = []


for patient_id, windows_rr_pat in tqdm(windows_rr.items(), desc="Create data for training", leave=False):

    # Add timestamps for each window (start and end times)
    win_start = np.asarray(windows_ts[patient_id][:, 0])
    win_end = np.asarray(windows_ts[patient_id][:, -1])
    timestamp = np.concatenate((win_start.reshape(-1, 1), win_end.reshape(-1, 1)), axis=1)

    num_windows = len(windows_rr_pat)

    # Sanity check
    if len(win_start) != num_windows or len(win_end) != num_windows:
        print(f"Length mismatch for patient_id={patient_id}, skipping.")
        continue

    # Add patient IDs
    patient_id_array = np.repeat(str(patient_id), num_windows).reshape(-1, 1)

    # Combine all features into one array per patient
    X_rec = np.concatenate((
        windows_rr_pat.astype(np.float32),
        prec_windows[patient_id].reshape(-1, 1),
        global_labels[patient_id].reshape(-1, 1),
        patient_id_array
    ), axis=1)

    X_all.append(X_rec)

    # Add true label for each patient
    y_all.append(np.asarray(combined_y_df.loc[combined_y_df.patient_id==patient_id, 'true_label']))

    timestamp_all.append(timestamp)

# Concatenate all patients
X_all = np.vstack(X_all)
timestamp_all = np.vstack(timestamp_all)
y_all = np.concatenate(tuple(y for y in y_all), axis=0)

Create data for training:   0%|          | 0/98 [00:00<?, ?it/s]

In [8]:
final_data = (X_all, y_all, timestamp_all)
training_file_path = os.path.join(output_path, 'training_data.pickle')
with open(training_file_path, 'wb') as f:
    pickle.dump(final_data, f)

print("Data for ArNet2 model prepared and saved as training_data.pickle")


Data for ArNet2 model prepared and saved as training_data.pickle


### 9. Train ArNet2 Model

Now that we have preprocessed the data, we will train the ArNet2 model.

In [9]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
config_file_path = os.path.join(project_root, 'config', 'config.yml')
import os
import yaml

# Assuming config.yml is in the ./config/ directory of your project
config_file_path = os.path.join(project_root, 'config', 'config.yml')

# Load the config file (YAML format assumed)
with open(config_file_path, 'r') as file:
    config = yaml.safe_load(file)

# Convert relative paths to absolute paths based on project_root
config['path']['arnet2'] = os.path.abspath(os.path.join(project_root, config['path']['arnet2']))
config['path']['resnet'] = os.path.abspath(os.path.join(project_root, config['path']['resnet']))

# Save the updated config back to the file
updated_config_file_path = os.path.join(project_root, 'config', 'config_w_abs_path.yml')

with open(updated_config_file_path, 'w') as file:
    yaml.dump(config, file, default_flow_style=False)

subprocess.run(['python', '../run_ArNet2.py', '--mode', 'train', '--input_file', training_file_path, '--output_name', 'ArNet2', '--save_model_path', '../model/', '--config', updated_config_file_path])

Training model...
169/169 [==============================] - 1s 3ms/step
Training GRU for label: 0
Epoch 1/5
35/35 [==============================] - 2s 9ms/step - loss: 0.3970 - accuracy: 0.9002 - auc_4: 0.0000e+00
Epoch 2/5
35/35 [==============================] - 0s 9ms/step - loss: 0.1483 - accuracy: 0.9990 - auc_4: 0.0000e+00
Epoch 3/5
35/35 [==============================] - 0s 9ms/step - loss: 0.1126 - accuracy: 1.0000 - auc_4: 0.0000e+00
Epoch 4/5
35/35 [==============================] - 0s 9ms/step - loss: 0.1100 - accuracy: 1.0000 - auc_4: 0.0000e+00
Epoch 5/5
35/35 [==============================] - 0s 9ms/step - loss: 0.1107 - accuracy: 1.0000 - auc_4: 0.0000e+00
Training GRU for label: 1
Epoch 1/5
120/120 [==============================] - 2s 9ms/step - loss: 0.4541 - accuracy: 0.8159 - auc_5: 0.8533
Epoch 2/5
120/120 [==============================] - 1s 9ms/step - loss: 0.3108 - accuracy: 0.8810 - auc_5: 0.9317
Epoch 3/5
120/120 [==============================] - 1s 9ms/

CompletedProcess(args=['python', '../run_ArNet2.py', '--mode', 'train', '--input_file', '.././data/training_data.pickle', '--output_name', 'ArNet2', '--save_model_path', '../model/', '--config', '/home/shanybiton/repos/Shany_Repo/plug-and-play/config/config_w_abs_path.yml'], returncode=0)

### 10. Summary

This notebook demonstrated the following steps:

- Preprocessing the SHDB-AF dataset (R-peak annotations, RR intervals, windows).

- Training and saving the ArNet2 model for AF classification.

You can now use the trained model to predict AF in ECG signals from new patients.